In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU - using CPU")

False
No GPU - using CPU


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_csv("../data/medquad_augmented.csv")

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["qtype"])

print(df[["qtype", "label"]].drop_duplicates().sort_values("label"))

                                        qtype  label
0                                      causes      0
3000                          exams and tests      1
6000                              information      2
9000                              inheritance      3
12000                                 outlook      4
15000                             precautions      5
18000                            side effects      6
21000                                symptoms      7
24000                               treatment      8
27000  when to contact a medical professional      9


In [5]:
import json

label_map = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
with open("../models/label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print(label_map)

{'causes': 0, 'exams and tests': 1, 'information': 2, 'inheritance': 3, 'outlook': 4, 'precautions': 5, 'side effects': 6, 'symptoms': 7, 'treatment': 8, 'when to contact a medical professional': 9}


In [6]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 21000
Validation: 4500
Test: 4500


In [7]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=10
)

print("Tokenizer and model loaded")

c:\Users\vvgk0135.DS\Desktop\mkdir medical-intent-classifier\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vvgk0135.DS\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11142.03it/s]
[tra

Tokenizer and model loaded


In [8]:
class MedicalIntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [9]:
train_dataset = MedicalIntentDataset(train_df["question"], train_df["label"], tokenizer)
val_dataset = MedicalIntentDataset(val_df["question"], val_df["label"], tokenizer)
test_dataset = MedicalIntentDataset(test_df["question"], test_df["label"], tokenizer)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 21000
Validation dataset size: 4500
Test dataset size: 4500


In [10]:
BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Number of training batches: {len(train_loader)}")

Number of training batches: 2625


In [11]:
tiny_train = MedicalIntentDataset(
    train_df["question"].iloc[:50],
    train_df["label"].iloc[:50],
    tokenizer
)
tiny_loader = DataLoader(tiny_train, batch_size=8, shuffle=True)

batch = next(iter(tiny_loader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8])
